In [1]:
from pathlib import Path
import pandas as pd

# Project root
PROJECT_ROOT = Path(r"D:\Nexzibyte Solution\nexzibyte-predictive-bi")

# Folders to inspect
FOLDERS_TO_CHECK = [
    PROJECT_ROOT / "notebooks",
    PROJECT_ROOT / "models",
    PROJECT_ROOT / "data" / "processed",
    PROJECT_ROOT / "data" / "synthetic",
    PROJECT_ROOT / "reports",
]

# Find existing files without modifying anything
records = []

for folder in FOLDERS_TO_CHECK:
    if not folder.exists():
        print(f"WARNING: Folder not found: {folder}")
        continue

    for file_path in sorted(folder.iterdir()):
        if file_path.is_file():
            records.append({
                "Folder": str(folder.relative_to(PROJECT_ROOT)),
                "File_Name": file_path.name,
                "Extension": file_path.suffix.lower(),
                "Size_KB": round(file_path.stat().st_size / 1024, 2),
            })

inventory = pd.DataFrame(records)

if inventory.empty:
    print("No files found. Please verify the project directory.")
else:
    print(f"Total files found: {len(inventory)}")

    display(
        inventory[
            inventory["File_Name"].str.contains(
                r"Day6|Day7|Day8|Day9|Day10|Day11|model|forecast|recommendation|anomaly|risk|customer",
                case=False,
                regex=True,
            )
        ].reset_index(drop=True)
    )

Total files found: 84


,Folder,File_Name,Extension,Size_KB
0,notebooks,04_lead_model.ipynb,.ipynb,41.15
1,notebooks,05_quote_model.ipynb,.ipynb,57.52
2,notebooks,06_revenue_forecast.ipynb,.ipynb,55.96
3,notebooks,07_project_risk.ipynb,.ipynb,63.32
4,notebooks,08_customer_intelligence.ipynb,.ipynb,100.37
...,...,...,...,...
62,reports,Day6_Lead_Model_Evaluation.md,.md,1.36
63,reports,Day7_Quote_Model_Evaluation.md,.md,4.67
64,reports,Day8_Revenue_Forecasting_Report.md,.md,4.58
65,reports,Day9_Margin_Risk_Report.md,.md,1.24


In [2]:
from pathlib import Path

PROJECT_ROOT = Path(r"D:\Nexzibyte Solution\nexzibyte-predictive-bi")
REPORTS_DIR = PROJECT_ROOT / "reports"

report_names = [
    "Day6_Lead_Model_Evaluation.md",
    "Day7_Quote_Model_Evaluation.md",
    "Day8_Revenue_Forecasting_Report.md",
    "Day9_Project_Risk_Summary.md",
    "Day9_Margin_Risk_Report.md",
    "Day10_Customer_Intelligence_Report.md",
    "Day11_Recommendation_Anomaly_Detection_Report.md",
]

for report_name in report_names:
    report_path = REPORTS_DIR / report_name

    print("\n" + "=" * 90)
    print(f"REPORT: {report_name}")
    print("=" * 90)

    if report_path.exists():
        content = report_path.read_text(encoding="utf-8")
        print(content)
    else:
        print("REPORT NOT FOUND")


REPORT: Day6_Lead_Model_Evaluation.md
# Day 6 — Lead Model Evaluation

## Data status
SYNTHETIC DEMONSTRATION ONLY. Outcome labels were randomly generated.
These metrics do not represent actual NexziByte lead-conversion performance.

## Dataset
- Total records: 180
- Training records: 135
- Test records: 45
- Input features: Industry / Sub-industry, Target Customers / B2B or B2C, Founded Year (Standardized), Employee Count (Numeric), Company Age (Years)
- Target: Synthetic_Lead_Conversion

## Logistic Regression baseline

| Metric | Value |
|---|---:|
| Accuracy | 0.6222 |
| Precision | 0.3333 |
| Recall | 0.0625 |
| F1-score | 0.1053 |
| ROC-AUC | 0.5172 |
| PR-AUC (Average Precision) | 0.4217 |

## Confusion matrix
Rows are actual labels [0, 1]; columns are predicted labels [0, 1].

[[27, 2], [15, 1]]

## Limitations
- No verified historical conversion outcome was available in the audited prospect dataset.
- Synthetic labels were randomly generated and do not represent real business

In [3]:
from pathlib import Path
import json
import importlib.util
import re

PROJECT_ROOT = Path(r"D:\Nexzibyte Solution\nexzibyte-predictive-bi")

NOTEBOOKS = [
    "04_lead_model.ipynb",
    "05_quote_model.ipynb",
    "06_revenue_forecast.ipynb",
    "07_project_risk.ipynb",
    "08_customer_intelligence.ipynb",
    "09_recommendations_anomaly_detection.ipynb",
]

# Check whether relevant libraries are installed.
print("=" * 75)
print("LIBRARY AVAILABILITY")
print("=" * 75)

for package in ["sklearn", "shap", "pandas", "numpy", "matplotlib"]:
    available = importlib.util.find_spec(package) is not None
    print(f"{package}: {'INSTALLED' if available else 'NOT INSTALLED'}")


# Inspect model-related code without executing or modifying notebooks.
print("\n" + "=" * 75)
print("MODEL CODE INVENTORY")
print("=" * 75)

patterns = {
    "Model definitions": r"\b(LogisticRegression|RandomForestClassifier|"
                         r"RandomForestRegressor|LinearRegression|"
                         r"IsolationForest|KMeans|GradientBoostingClassifier|"
                         r"GradientBoostingRegressor)\b",
    "Model fitting": r"\.fit\s*\(",
    "Probability predictions": r"\.predict_proba\s*\(",
    "Predictions": r"\.predict\s*\(",
    "Feature importance": r"feature_importances_|coef_",
    "Calibration": r"CalibratedClassifierCV|calibration_curve|"
                   r"brier_score_loss|CalibratedClassifier",
    "SHAP": r"\bshap\b",
}

for notebook_name in NOTEBOOKS:
    notebook_path = PROJECT_ROOT / "notebooks" / notebook_name

    print("\n" + "-" * 75)
    print(f"NOTEBOOK: {notebook_name}")
    print("-" * 75)

    if not notebook_path.exists():
        print("NOTEBOOK NOT FOUND")
        continue

    with notebook_path.open("r", encoding="utf-8") as file:
        notebook = json.load(file)

    source_cells = [
        "".join(cell.get("source", []))
        for cell in notebook.get("cells", [])
        if cell.get("cell_type") == "code"
    ]

    source = "\n".join(source_cells)

    for label, pattern in patterns.items():
        matches = sorted(set(re.findall(pattern, source, flags=re.IGNORECASE)))
        print(f"{label}: {', '.join(matches) if matches else 'Not detected'}")

LIBRARY AVAILABILITY
sklearn: INSTALLED
shap: NOT INSTALLED
pandas: INSTALLED
numpy: INSTALLED
matplotlib: INSTALLED

MODEL CODE INVENTORY

---------------------------------------------------------------------------
NOTEBOOK: 04_lead_model.ipynb
---------------------------------------------------------------------------
Model definitions: LogisticRegression
Model fitting: .fit(
Probability predictions: .predict_proba(
Predictions: .predict(
Feature importance: Not detected
Calibration: Not detected
SHAP: Not detected

---------------------------------------------------------------------------
NOTEBOOK: 05_quote_model.ipynb
---------------------------------------------------------------------------
Model definitions: LogisticRegression
Model fitting: .fit(
Probability predictions: .predict_proba(
Predictions: .predict(
Feature importance: coef_
Calibration: Not detected
SHAP: SHAP

---------------------------------------------------------------------------
NOTEBOOK: 06_revenue_forecast.

In [4]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(r"D:\Nexzibyte Solution\nexzibyte-predictive-bi")

output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

model_inventory = [
    {
        "Model_ID": "DAY6_LEAD_LOGISTIC",
        "Project_Day": 6,
        "Use_Case": "Lead classification",
        "Notebook": "04_lead_model.ipynb",
        "Model_Type": "Logistic Regression",
        "Target": "Synthetic lead label",
        "Data_Type": "Synthetic demonstration",
        "Explainability_Method": "Logistic regression coefficients",
        "Probability_Output": "Yes",
        "Calibration_Status": "Not assessed",
        "Production_Recommendation": "Not approved",
        "Reason": (
            "Synthetic labels; historical conversion outcomes are unavailable. "
            "Existing test ROC-AUC is approximately 0.5172."
        ),
    },
    {
        "Model_ID": "DAY7_QUOTE_LOGISTIC",
        "Project_Day": 7,
        "Use_Case": "Quotation win classification",
        "Notebook": "05_quote_model.ipynb",
        "Model_Type": "Logistic Regression",
        "Target": "Won versus Lost quotation",
        "Data_Type": "Synthetic demonstration",
        "Explainability_Method": "Logistic regression coefficients",
        "Probability_Output": "Yes",
        "Calibration_Status": "Not assessed",
        "Production_Recommendation": "Not approved",
        "Reason": (
            "Synthetic quotation records; coefficients are associations in "
            "synthetic data, not verified business drivers. Calibration is pending."
        ),
    },
    {
        "Model_ID": "DAY8_REVENUE_LINEAR",
        "Project_Day": 8,
        "Use_Case": "Revenue forecasting",
        "Notebook": "06_revenue_forecast.ipynb",
        "Model_Type": "Linear Regression",
        "Target": "Synthetic monthly revenue",
        "Data_Type": "Synthetic demonstration",
        "Explainability_Method": "Linear regression coefficients",
        "Probability_Output": "No",
        "Calibration_Status": "Not applicable to point forecasts",
        "Production_Recommendation": "Not approved",
        "Reason": (
            "Only 36 synthetic monthly observations; the naive forecast "
            "performed best on the existing six-month synthetic holdout."
        ),
    },
    {
        "Model_ID": "DAY9_PROJECT_DELAY_RF",
        "Project_Day": 9,
        "Use_Case": "Project delay risk",
        "Notebook": "07_project_risk.ipynb",
        "Model_Type": "Random Forest Classifier",
        "Target": "Synthetic project delay label",
        "Data_Type": "Synthetic demonstration",
        "Explainability_Method": "Random Forest feature importance",
        "Probability_Output": "Yes",
        "Calibration_Status": "Not assessed",
        "Production_Recommendation": "Not approved",
        "Reason": (
            "Synthetic project records. Feature importance must be verified "
            "against the fitted model and its actual feature columns."
        ),
    },
    {
        "Model_ID": "DAY9_EFFORT_OVERRUN_LOGISTIC",
        "Project_Day": 9,
        "Use_Case": "Project effort-overrun risk",
        "Notebook": "07_project_risk.ipynb",
        "Model_Type": "Logistic Regression",
        "Target": "Synthetic effort-overrun label",
        "Data_Type": "Synthetic demonstration",
        "Explainability_Method": "Logistic regression coefficients",
        "Probability_Output": "Yes",
        "Calibration_Status": "Not assessed",
        "Production_Recommendation": "Not approved",
        "Reason": (
            "Synthetic project records. Coefficients show model associations, "
            "not causal explanations."
        ),
    },
    {
        "Model_ID": "DAY9_LOW_MARGIN_MODELS",
        "Project_Day": 9,
        "Use_Case": "Low-margin risk",
        "Notebook": "07_project_risk.ipynb",
        "Model_Type": "Logistic Regression and Random Forest",
        "Target": "Synthetic low-margin risk label",
        "Data_Type": "Synthetic demonstration",
        "Explainability_Method": "Coefficients and feature importance, if available",
        "Probability_Output": "Yes",
        "Calibration_Status": "Not assessed",
        "Production_Recommendation": "Do not use for risk decisions",
        "Reason": (
            "Both evaluated models performed poorly on the synthetic test set; "
            "reported ROC-AUC values were below 0.50."
        ),
    },
    {
        "Model_ID": "DAY10_CUSTOMER_INTELLIGENCE",
        "Project_Day": 10,
        "Use_Case": "Customer segmentation and recommendations",
        "Notebook": "08_customer_intelligence.ipynb",
        "Model_Type": "Rule-based analysis",
        "Target": "Descriptive segments and provisional inactivity proxy",
        "Data_Type": "Synthetic demonstration",
        "Explainability_Method": "Documented rules and thresholds",
        "Probability_Output": "No",
        "Calibration_Status": "Not applicable",
        "Production_Recommendation": "Business validation required",
        "Reason": (
            "No validated churn model was trained. Customer lifetime value "
            "cannot be calculated reliably from the available data."
        ),
    },
    {
        "Model_ID": "DAY11_RECOMMENDATIONS_ANOMALIES",
        "Project_Day": 11,
        "Use_Case": "Customer recommendations and anomaly screening",
        "Notebook": "09_recommendations_anomaly_detection.ipynb",
        "Model_Type": "Rule-based engine",
        "Target": "Recommended actions and rule-triggered flags",
        "Data_Type": "Synthetic demonstration",
        "Explainability_Method": "Documented business rules and thresholds",
        "Probability_Output": "No",
        "Calibration_Status": "Not applicable",
        "Production_Recommendation": "Business validation required",
        "Reason": (
            "Thresholds are provisional. Rule-triggered flags are screening "
            "results, not proof of fraud, error, or commercial risk."
        ),
    },
]

inventory_df = pd.DataFrame(model_inventory)

output_path = output_dir / "Day12_Model_Inventory.csv"
inventory_df.to_csv(output_path, index=False)

print("DAY 12 MODEL INVENTORY")
print("=" * 75)
print(f"Models and analytical engines documented: {len(inventory_df)}")
print(f"Output saved to: {output_path}")
print("\nModel types:")
print(inventory_df["Model_Type"].value_counts().to_string())
print("\nProduction approval status:")
print(inventory_df["Production_Recommendation"].value_counts().to_string())

assert output_path.exists()
assert len(inventory_df) == 8
assert inventory_df["Model_ID"].is_unique

print("\nPASS: Model inventory saved and validation completed.")

DAY 12 MODEL INVENTORY
Models and analytical engines documented: 8
Output saved to: D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\processed\Day12_Model_Inventory.csv

Model types:
Model_Type
Logistic Regression                      3
Linear Regression                        1
Random Forest Classifier                 1
Logistic Regression and Random Forest    1
Rule-based analysis                      1
Rule-based engine                        1

Production approval status:
Production_Recommendation
Not approved                     5
Business validation required     2
Do not use for risk decisions    1

PASS: Model inventory saved and validation completed.


In [5]:
from pathlib import Path
import json
import re

PROJECT_ROOT = Path(r"D:\Nexzibyte Solution\nexzibyte-predictive-bi")

notebook_names = [
    "04_lead_model.ipynb",
    "05_quote_model.ipynb",
    "06_revenue_forecast.ipynb",
    "07_project_risk.ipynb",
]

patterns = {
    "Target assignment": r"\b(?:y|y_train|y_test|target|target_col)\s*=",
    "Feature assignment": r"\b(?:X|X_train|X_test|features|feature_cols)\s*=",
    "Model creation": r"\b(?:model|clf|classifier|regressor)\s*=\s*",
    "Model fitting": r"\.fit\s*\(",
    "Prediction": r"\.predict\s*\(",
    "Probability prediction": r"\.predict_proba\s*\(",
    "Feature names": r"feature_names_in_",
    "Feature importance": r"feature_importances_",
    "Coefficients": r"coef_",
    "Calibration metrics": r"brier_score_loss|calibration_curve|CalibratedClassifierCV",
}

for notebook_name in notebook_names:
    notebook_path = PROJECT_ROOT / "notebooks" / notebook_name

    print("\n" + "=" * 80)
    print(f"NOTEBOOK: {notebook_name}")
    print("=" * 80)

    if not notebook_path.exists():
        print("Notebook not found.")
        continue

    with notebook_path.open("r", encoding="utf-8") as file:
        notebook = json.load(file)

    for index, cell in enumerate(notebook.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        matched = [
            label
            for label, pattern in patterns.items()
            if re.search(pattern, source)
        ]

        if matched:
            print(f"\nCell {index}: {', '.join(matched)}")
            print("-" * 60)
            print(source[:3500])

            if len(source) > 3500:
                print("... [cell output truncated for inspection]")


NOTEBOOK: 04_lead_model.ipynb

Cell 3: Target assignment, Feature assignment
------------------------------------------------------------
# Day 6 - Step 5: Prepare train/test split

# Exclude identifiers, existing component scores, and data-quality flags
# to avoid learning the existing scoring system instead of conversion.
categorical_features = [
    "Industry / Sub-industry",
    "Target Customers / B2B or B2C"
]

numeric_features = [
    "Founded Year (Standardized)",
    "Employee Count (Numeric)",
    "Company Age (Years)"
]

feature_cols = categorical_features + numeric_features
target_col = "Synthetic_Lead_Conversion"

X = model_df[feature_cols].copy()
y = model_df[target_col].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts().sort_index())

p

In [6]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import brier_score_loss, log_loss


# ============================================================
# DAY 12 - STEP 6
# EXPLAINABILITY AND PROBABILITY CALIBRATION
# SYNTHETIC DATA ONLY
# ============================================================

PROJECT_ROOT = Path(
    r"D:\Nexzibyte Solution\nexzibyte-predictive-bi"
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SYNTHETIC_DIR = PROJECT_ROOT / "data" / "synthetic"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

explainability_rows = []
calibration_rows = []


def record_explanations(
    model_id,
    fitted_pipeline,
    feature_names,
    method
):
    """Save model-specific feature explanations."""

    estimator = fitted_pipeline.steps[-1][1]

    if hasattr(estimator, "coef_"):
        values = estimator.coef_[0]
        value_name = "Coefficient"

        for feature, value in zip(feature_names, values):
            explainability_rows.append({
                "Model_ID": model_id,
                "Feature": feature,
                "Explanation_Method": method,
                "Explanation_Value": float(value),
                "Value_Type": value_name,
                "Interpretation": (
                    "Positive coefficients increase estimated log-odds "
                    "of class 1; negative coefficients decrease them. "
                    "This is an association, not causation."
                ),
                "Data_Status": "Synthetic demonstration only",
            })

    elif hasattr(estimator, "feature_importances_"):
        values = estimator.feature_importances_
        value_name = "Feature Importance"

        for feature, value in zip(feature_names, values):
            explainability_rows.append({
                "Model_ID": model_id,
                "Feature": feature,
                "Explanation_Method": method,
                "Explanation_Value": float(value),
                "Value_Type": value_name,
                "Interpretation": (
                    "Relative impurity-based importance. "
                    "This is not a causal effect or a direction of impact."
                ),
                "Data_Status": "Synthetic demonstration only",
            })


def record_calibration(model_id, y_true, probabilities):
    """Record probability scores and observed outcome rates."""

    probabilities = np.asarray(probabilities, dtype=float)
    y_true = np.asarray(y_true, dtype=int)

    brier = brier_score_loss(y_true, probabilities)
    logloss = log_loss(
        y_true,
        np.column_stack([1 - probabilities, probabilities]),
        labels=[0, 1]
    )

    calibration_rows.append({
        "Model_ID": model_id,
        "Test_Records": len(y_true),
        "Observed_Positive_Rate": float(y_true.mean()),
        "Mean_Predicted_Probability": float(probabilities.mean()),
        "Brier_Score": float(brier),
        "Log_Loss": float(logloss),
        "Data_Status": "Synthetic demonstration only",
        "Interpretation": (
            "Descriptive evaluation only. Lower Brier score and log loss "
            "are generally better when comparing models on the same data "
            "and target. These metrics alone do not establish calibration."
        ),
    })

    # Five equal-width probability bins.
    bins = np.linspace(0, 1, 6)
    bin_ids = np.digitize(
        probabilities,
        bins[1:-1],
        right=False
    )

    for bin_id in range(5):
        mask = bin_ids == bin_id

        if not mask.any():
            continue

        calibration_rows.append({
            "Model_ID": model_id,
            "Test_Records": int(mask.sum()),
            "Observed_Positive_Rate": float(y_true[mask].mean()),
            "Mean_Predicted_Probability": float(
                probabilities[mask].mean()
            ),
            "Brier_Score": np.nan,
            "Log_Loss": np.nan,
            "Data_Status": "Synthetic demonstration only",
            "Interpretation": (
                f"Calibration bin {bin_id + 1}: "
                f"{bins[bin_id]:.1f} to {bins[bin_id + 1]:.1f}. "
                "Small bin counts can make the observed rate unstable."
            ),
        })


# ------------------------------------------------------------
# A. QUOTATION WIN MODEL
# ------------------------------------------------------------

quote_path = (
    SYNTHETIC_DIR / "Day7_Synthetic_Quotation_Dataset.csv"
)

quotes = pd.read_csv(quote_path)

quote_model_df = quotes[
    quotes["Quote_Status"].isin(["Won", "Lost"])
].copy()

quote_features = [
    "Quote_Value",
    "Discount_Pct",
    "Revision_Count",
    "Engagement_Count",
]

X = quote_model_df[quote_features].copy()
y = (
    quote_model_df["Quote_Status"] == "Won"
).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

quote_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        random_state=42,
        max_iter=1000,
    )),
])

quote_model.fit(X_train, y_train)

quote_probabilities = quote_model.predict_proba(X_test)[:, 1]

record_explanations(
    "DAY7_QUOTE_LOGISTIC",
    quote_model,
    quote_features,
    "Standardized logistic regression coefficients",
)

record_calibration(
    "DAY7_QUOTE_LOGISTIC",
    y_test,
    quote_probabilities,
)


# ------------------------------------------------------------
# B. PROJECT RISK MODELS
# ------------------------------------------------------------

risk_path = (
    SYNTHETIC_DIR / "Day9_Synthetic_Project_Risk_Dataset.csv"
)

risk_df = pd.read_csv(
    risk_path,
    parse_dates=["Project_Start_Date", "Data_Cutoff_Date"],
)

risk_features = [
    "Planned_Effort",
    "Milestones_Planned",
    "Milestones_Completed",
    "Change_Request_Count",
    "Dependency_Delay_Days",
    "Client_Delay_Days",
    "Resource_Utilization",
    "Project_Age_Days_At_Cutoff",
]

risk_df = risk_df.sort_values(
    ["Project_Start_Date", "Project_ID"]
).reset_index(drop=True)

split_index = int(len(risk_df) * 0.80)

train_df = risk_df.iloc[:split_index].copy()
test_df = risk_df.iloc[split_index:].copy()

X_train = train_df[risk_features]
X_test = test_df[risk_features]


# Project delay: Random Forest
delay_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
    )),
])

delay_model.fit(
    X_train,
    train_df["Project_Delayed"].astype(int),
)

delay_probabilities = delay_model.predict_proba(X_test)[:, 1]

record_explanations(
    "DAY9_PROJECT_DELAY_RF",
    delay_model,
    risk_features,
    "Random Forest impurity-based feature importance",
)

record_calibration(
    "DAY9_PROJECT_DELAY_RF",
    test_df["Project_Delayed"].astype(int),
    delay_probabilities,
)


# Effort overrun: Logistic Regression
effort_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42,
    )),
])

effort_model.fit(
    X_train,
    train_df["Effort_Overrun"].astype(int),
)

effort_probabilities = effort_model.predict_proba(X_test)[:, 1]

record_explanations(
    "DAY9_EFFORT_OVERRUN_LOGISTIC",
    effort_model,
    risk_features,
    "Standardized logistic regression coefficients",
)

record_calibration(
    "DAY9_EFFORT_OVERRUN_LOGISTIC",
    test_df["Effort_Overrun"].astype(int),
    effort_probabilities,
)


# ------------------------------------------------------------
# C. SAVE OUTPUTS
# ------------------------------------------------------------

explainability_df = pd.DataFrame(explainability_rows)

explainability_path = (
    PROCESSED_DIR / "Day12_Model_Explainability.csv"
)

explainability_df.to_csv(
    explainability_path,
    index=False,
)

calibration_df = pd.DataFrame(calibration_rows)

calibration_path = (
    PROCESSED_DIR / "Day12_Probability_Calibration_Checks.csv"
)

calibration_df.to_csv(
    calibration_path,
    index=False,
)


# ------------------------------------------------------------
# D. VALIDATION
# ------------------------------------------------------------

print("=" * 75)
print("DAY 12: EXPLAINABILITY AND CALIBRATION")
print("=" * 75)

print("\nEXPLAINABILITY SUMMARY")
print(
    explainability_df.groupby(
        ["Model_ID", "Value_Type"]
    ).size().to_string()
)

print("\nOVERALL CALIBRATION METRICS")
overall_metrics = calibration_df[
    calibration_df["Brier_Score"].notna()
]

print(
    overall_metrics[
        [
            "Model_ID",
            "Test_Records",
            "Observed_Positive_Rate",
            "Mean_Predicted_Probability",
            "Brier_Score",
            "Log_Loss",
        ]
    ].to_string(index=False)
)

print("\nSAVED FILES")
print(explainability_path)
print(calibration_path)

assert len(explainability_df) == 20
assert explainability_df["Explanation_Value"].notna().all()
assert len(overall_metrics) == 3
assert calibration_path.exists()
assert explainability_path.exists()

print("\nPASS: Explainability and calibration outputs created.")
print("Reminder: all model results use synthetic data.")

DAY 12: EXPLAINABILITY AND CALIBRATION

EXPLAINABILITY SUMMARY
Model_ID                      Value_Type        
DAY7_QUOTE_LOGISTIC           Coefficient           4
DAY9_EFFORT_OVERRUN_LOGISTIC  Coefficient           8
DAY9_PROJECT_DELAY_RF         Feature Importance    8

OVERALL CALIBRATION METRICS
                    Model_ID  Test_Records  Observed_Positive_Rate  Mean_Predicted_Probability  Brier_Score  Log_Loss
         DAY7_QUOTE_LOGISTIC            51                0.470588                    0.473182     0.228812  0.649863
       DAY9_PROJECT_DELAY_RF            60                0.916667                    0.653228     0.140460  0.444066
DAY9_EFFORT_OVERRUN_LOGISTIC            60                0.450000                    0.443293     0.148060  0.444191

SAVED FILES
D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\processed\Day12_Model_Explainability.csv
D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\processed\Day12_Probability_Calibration_Checks.csv

PASS: Explainabil

In [7]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    r"D:\Nexzibyte Solution\nexzibyte-predictive-bi"
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

calibration_path = (
    PROCESSED_DIR / "Day12_Probability_Calibration_Checks.csv"
)

explainability_path = (
    PROCESSED_DIR / "Day12_Model_Explainability.csv"
)

calibration_df = pd.read_csv(calibration_path)
explainability_df = pd.read_csv(explainability_path)

overall_metrics = calibration_df[
    calibration_df["Brier_Score"].notna()
].copy()

model_definitions = [
    {
        "Model_ID": "DAY7_QUOTE_LOGISTIC",
        "Model_Name": "Quotation Win Classification",
        "Purpose": "Demonstrate classification of synthetic Won/Lost quotations.",
        "Model_Type": "Logistic Regression",
        "Features": "Quote_Value, Discount_Pct, Revision_Count, Engagement_Count",
        "Target": "Won versus Lost; Pending quotations excluded.",
        "Split": "Stratified random split, 80% train and 20% test, random_state=42.",
        "Explanation": "Standardized logistic regression coefficients.",
        "Intended_Use": "Educational evaluation and model-governance demonstration only.",
        "Prohibited_Use": "Do not use to approve discounts, prioritize real quotations, or forecast actual sales.",
        "Limitations": "Synthetic data and outcomes; no evidence of real-world validity.",
    },
    {
        "Model_ID": "DAY9_PROJECT_DELAY_RF",
        "Model_Name": "Project Delay Risk",
        "Purpose": "Demonstrate classification of synthetic project-delay labels.",
        "Model_Type": "Random Forest Classifier",
        "Features": "Planned_Effort, Milestones_Planned, Milestones_Completed, Change_Request_Count, Dependency_Delay_Days, Client_Delay_Days, Resource_Utilization, Project_Age_Days_At_Cutoff",
        "Target": "Project_Delayed",
        "Split": "Chronological 80% train and 20% test split.",
        "Explanation": "Random Forest impurity-based feature importance.",
        "Intended_Use": "Educational risk-analysis and governance demonstration only.",
        "Prohibited_Use": "Do not make real project staffing, escalation, or delivery decisions from these probabilities.",
        "Limitations": "Synthetic data. Test positive rate is 91.67%, compared with a mean predicted probability of 65.32%; probability reliability needs further investigation.",
    },
    {
        "Model_ID": "DAY9_EFFORT_OVERRUN_LOGISTIC",
        "Model_Name": "Project Effort Overrun Risk",
        "Purpose": "Demonstrate classification of synthetic effort-overrun labels.",
        "Model_Type": "Logistic Regression",
        "Features": "Planned_Effort, Milestones_Planned, Milestones_Completed, Change_Request_Count, Dependency_Delay_Days, Client_Delay_Days, Resource_Utilization, Project_Age_Days_At_Cutoff",
        "Target": "Effort_Overrun",
        "Split": "Chronological 80% train and 20% test split.",
        "Explanation": "Standardized logistic regression coefficients.",
        "Intended_Use": "Educational risk-analysis and governance demonstration only.",
        "Prohibited_Use": "Do not use to estimate actual project costs, staffing requirements, or contractual exposure.",
        "Limitations": "Synthetic data; coefficients represent model associations, not causal effects. Probability calibration is not yet established.",
    },
]

cards = []

for definition in model_definitions:
    model_id = definition["Model_ID"]

    metric_rows = overall_metrics[
        overall_metrics["Model_ID"] == model_id
    ]

    if metric_rows.empty:
        raise ValueError(
            f"Calibration metrics missing for {model_id}"
        )

    metrics = metric_rows.iloc[0]

    model_explanations = explainability_df[
        explainability_df["Model_ID"] == model_id
    ]

    top_features = model_explanations.copy()

    if not top_features.empty:
        top_features["Absolute_Value"] = (
            top_features["Explanation_Value"].abs()
        )

        top_features = top_features.sort_values(
            "Absolute_Value",
            ascending=False,
        )

        top_feature_names = ", ".join(
            top_features["Feature"].head(5).tolist()
        )
    else:
        top_feature_names = "Not available"

    card = {
        **definition,
        "Test_Records": int(metrics["Test_Records"]),
        "Observed_Positive_Rate": float(
            metrics["Observed_Positive_Rate"]
        ),
        "Mean_Predicted_Probability": float(
            metrics["Mean_Predicted_Probability"]
        ),
        "Brier_Score": float(metrics["Brier_Score"]),
        "Log_Loss": float(metrics["Log_Loss"]),
        "Top_Explanation_Features": top_feature_names,
        "Data_Status": "Synthetic demonstration only",
        "Calibration_Status": "Exploratory check completed; reliability not established.",
        "Approval_Status": "Not approved for production use",
        "Governance_Status": "Business validation pending",
    }

    cards.append(card)

cards_df = pd.DataFrame(cards)

cards_path = PROCESSED_DIR / "Day12_Model_Cards.csv"
cards_df.to_csv(cards_path, index=False)

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

report_path = REPORTS_DIR / "Day12_Model_Cards_Report.md"

lines = [
    "# Day 12 — Explainable AI and Model Governance",
    "",
    "## Purpose",
    "",
    "Document model purpose, features, evaluation, explanations, limitations, "
    "and approval status.",
    "",
    "## Data and governance statement",
    "",
    "**All models covered here use synthetic demonstration data.** "
    "The results do not establish real-world predictive performance. "
    "No model is approved for production use.",
    "",
]

for card in cards:
    lines.extend([
        f"## {card['Model_Name']}",
        "",
        f"- **Model ID:** {card['Model_ID']}",
        f"- **Purpose:** {card['Purpose']}",
        f"- **Model type:** {card['Model_Type']}",
        f"- **Features:** {card['Features']}",
        f"- **Target:** {card['Target']}",
        f"- **Evaluation split:** {card['Split']}",
        f"- **Test records:** {card['Test_Records']}",
        f"- **Observed positive rate:** {card['Observed_Positive_Rate']:.4f}",
        f"- **Mean predicted probability:** {card['Mean_Predicted_Probability']:.4f}",
        f"- **Brier score:** {card['Brier_Score']:.4f}",
        f"- **Log loss:** {card['Log_Loss']:.4f}",
        f"- **Explanation method:** {card['Explanation']}",
        f"- **Top explanation features by absolute value:** {card['Top_Explanation_Features']}",
        f"- **Calibration status:** {card['Calibration_Status']}",
        f"- **Intended use:** {card['Intended_Use']}",
        f"- **Prohibited use:** {card['Prohibited_Use']}",
        f"- **Limitations:** {card['Limitations']}",
        f"- **Approval status:** {card['Approval_Status']}",
        "",
    ])

lines.extend([
    "## Governance requirements before real-world use",
    "",
    "1. Obtain verified, representative historical business data.",
    "2. Confirm target definitions and feature availability at prediction time.",
    "3. Evaluate performance on a suitable held-out dataset.",
    "4. Investigate probability calibration and class imbalance.",
    "5. Review fairness, data quality, privacy, and operational risks where applicable.",
    "6. Obtain business-owner approval and document monitoring responsibilities.",
    "",
    "## Interpretation warning",
    "",
    "Coefficients and feature importance describe how a fitted model behaves; "
    "they do not establish causation. A close match between the average predicted "
    "probability and observed outcome rate does not, by itself, prove calibration.",
    "",
])

report_path.write_text(
    "\n".join(lines),
    encoding="utf-8",
)

assert len(cards_df) == 3
assert cards_df["Model_ID"].is_unique
assert cards_df["Approval_Status"].eq(
    "Not approved for production use"
).all()
assert cards_path.exists()
assert report_path.exists()

print("DAY 12 MODEL CARDS")
print("=" * 75)
print("Model cards created:", len(cards_df))
print("CSV:", cards_path)
print("Report:", report_path)
print("\nModel card summary:")

print(
    cards_df[
        [
            "Model_Name",
            "Test_Records",
            "Brier_Score",
            "Calibration_Status",
            "Approval_Status",
        ]
    ].to_string(index=False)
)

print("\nPASS: Model cards and report created.")

DAY 12 MODEL CARDS
Model cards created: 3
CSV: D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\processed\Day12_Model_Cards.csv
Report: D:\Nexzibyte Solution\nexzibyte-predictive-bi\reports\Day12_Model_Cards_Report.md

Model card summary:
                  Model_Name  Test_Records  Brier_Score                                        Calibration_Status                 Approval_Status
Quotation Win Classification            51     0.228812 Exploratory check completed; reliability not established. Not approved for production use
          Project Delay Risk            60     0.140460 Exploratory check completed; reliability not established. Not approved for production use
 Project Effort Overrun Risk            60     0.148060 Exploratory check completed; reliability not established. Not approved for production use

PASS: Model cards and report created.


In [8]:
from pathlib import Path
from datetime import datetime
import hashlib
import json
import platform

import pandas as pd
import numpy as np
import sklearn


# ============================================================
# DAY 12 - STEP 9
# REPRODUCIBLE EXPERIMENT LOG
# ============================================================

PROJECT_ROOT = Path(
    r"D:\Nexzibyte Solution\nexzibyte-predictive-bi"
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 1. Source datasets and checksums
# ------------------------------------------------------------

source_files = {
    "Lead model": (
        PROJECT_ROOT / "data" / "processed"
        / "Day5_Feature_Dataset.csv"
    ),
    "Quotation model": (
        PROJECT_ROOT / "data" / "synthetic"
        / "Day7_Synthetic_Quotation_Dataset.csv"
    ),
    "Project risk models": (
        PROJECT_ROOT / "data" / "synthetic"
        / "Day9_Synthetic_Project_Risk_Dataset.csv"
    ),
    "Revenue forecast": (
        PROJECT_ROOT / "data" / "synthetic"
        / "Day8_Synthetic_Monthly_Revenue.csv"
    ),
}


def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(65536), b""):
            digest.update(chunk)

    return digest.hexdigest()


source_metadata = {}

for label, path in source_files.items():
    if path.exists():
        source_metadata[label] = {
            "path": str(path),
            "sha256": sha256_file(path),
            "exists": True,
        }
    else:
        source_metadata[label] = {
            "path": str(path),
            "sha256": None,
            "exists": False,
        }


# ------------------------------------------------------------
# 2. Read Day 12 outputs
# ------------------------------------------------------------

explainability_path = (
    PROCESSED_DIR / "Day12_Model_Explainability.csv"
)

calibration_path = (
    PROCESSED_DIR / "Day12_Probability_Calibration_Checks.csv"
)

model_cards_path = (
    PROCESSED_DIR / "Day12_Model_Cards.csv"
)

inventory_path = (
    PROCESSED_DIR / "Day12_Model_Inventory.csv"
)

required_paths = [
    explainability_path,
    calibration_path,
    model_cards_path,
    inventory_path,
]

missing = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "Required Day 12 output files are missing:\n"
        + "\n".join(missing)
    )

explainability_df = pd.read_csv(explainability_path)
calibration_df = pd.read_csv(calibration_path)
model_cards_df = pd.read_csv(model_cards_path)
inventory_df = pd.read_csv(inventory_path)

overall_calibration = calibration_df[
    calibration_df["Brier_Score"].notna()
]


# ------------------------------------------------------------
# 3. Build experiment records
# ------------------------------------------------------------

experiments = [
    {
        "Experiment_ID": "DAY12-EXP-001",
        "Experiment": "Quotation win model explainability",
        "Model_ID": "DAY7_QUOTE_LOGISTIC",
        "Notebook": "05_quote_model.ipynb",
        "Data_Source": source_metadata["Quotation model"]["path"],
        "Method": "Logistic Regression coefficients",
        "Split_Configuration": (
            "80/20 stratified random split; random_state=42"
        ),
        "Result_Summary": (
            "Four standardized coefficients generated."
        ),
        "Reproducibility_Seed": 42,
        "Status": "Completed",
    },
    {
        "Experiment_ID": "DAY12-EXP-002",
        "Experiment": "Project delay model explainability",
        "Model_ID": "DAY9_PROJECT_DELAY_RF",
        "Notebook": "07_project_risk.ipynb",
        "Data_Source": source_metadata["Project risk models"]["path"],
        "Method": "Random Forest impurity-based feature importance",
        "Split_Configuration": (
            "Chronological 80/20 split; random_state=42"
        ),
        "Result_Summary": (
            "Eight feature importance values generated. "
            "Probability reliability requires further assessment."
        ),
        "Reproducibility_Seed": 42,
        "Status": "Completed; calibration concern documented",
    },
    {
        "Experiment_ID": "DAY12-EXP-003",
        "Experiment": "Project effort-overrun explainability",
        "Model_ID": "DAY9_EFFORT_OVERRUN_LOGISTIC",
        "Notebook": "07_project_risk.ipynb",
        "Data_Source": source_metadata["Project risk models"]["path"],
        "Method": "Logistic Regression coefficients",
        "Split_Configuration": (
            "Chronological 80/20 split; random_state=42"
        ),
        "Result_Summary": (
            "Eight standardized coefficients generated."
        ),
        "Reproducibility_Seed": 42,
        "Status": "Completed",
    },
    {
        "Experiment_ID": "DAY12-EXP-004",
        "Experiment": "Probability calibration assessment",
        "Model_ID": "THREE_CLASSIFIERS",
        "Notebook": "10_explainable_ai_model_governance.ipynb",
        "Data_Source": (
            "Synthetic quotation and project-risk datasets"
        ),
        "Method": "Brier score, log loss, and five probability bins",
        "Split_Configuration": (
            "Existing model-specific test splits"
        ),
        "Result_Summary": (
            "Three classifiers assessed. Calibration reliability "
            "not established; project-delay mean probability "
            "was below its observed positive rate."
        ),
        "Reproducibility_Seed": 42,
        "Status": "Exploratory assessment completed",
    },
    {
        "Experiment_ID": "DAY12-EXP-005",
        "Experiment": "Model card and governance documentation",
        "Model_ID": "THREE_CLASSIFIERS",
        "Notebook": "10_explainable_ai_model_governance.ipynb",
        "Data_Source": "Day 12 explainability and calibration outputs",
        "Method": "Structured model cards and limitations review",
        "Split_Configuration": "Uses recorded model evaluation results",
        "Result_Summary": (
            "Three model cards created; none approved for production."
        ),
        "Reproducibility_Seed": None,
        "Status": "Completed",
    },
]

experiment_df = pd.DataFrame(experiments)

# Attach the recorded metric values to the relevant experiment.
experiment_df["Test_Records"] = np.nan
experiment_df["Brier_Score"] = np.nan
experiment_df["Log_Loss"] = np.nan
experiment_df["Data_Status"] = "Synthetic demonstration only"

for index, row in experiment_df.iterrows():
    matching = overall_calibration[
        overall_calibration["Model_ID"] == row["Model_ID"]
    ]

    if len(matching) == 1:
        experiment_df.loc[index, "Test_Records"] = int(
            matching.iloc[0]["Test_Records"]
        )
        experiment_df.loc[index, "Brier_Score"] = float(
            matching.iloc[0]["Brier_Score"]
        )
        experiment_df.loc[index, "Log_Loss"] = float(
            matching.iloc[0]["Log_Loss"]
        )


# ------------------------------------------------------------
# 4. Save experiment log and reproducibility manifest
# ------------------------------------------------------------

experiment_path = (
    PROCESSED_DIR / "Day12_Experiment_Log.csv"
)

experiment_df.to_csv(
    experiment_path,
    index=False,
)

manifest = {
    "project": "NexziByte Predictive Business Intelligence & Growth Engine",
    "project_day": 12,
    "created_at_local": datetime.now().astimezone().isoformat(),
    "python_version": platform.python_version(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "scikit_learn_version": sklearn.__version__,
    "random_seed_used_in_models": 42,
    "data_policy": (
        "Synthetic demonstration data; results are not "
        "validated business performance."
    ),
    "source_files": source_metadata,
    "generated_outputs": {
        "model_inventory": str(inventory_path),
        "model_explainability": str(explainability_path),
        "probability_calibration": str(calibration_path),
        "model_cards": str(model_cards_path),
        "experiment_log": str(experiment_path),
    },
    "governance_status": (
        "Technical documentation completed; business validation pending."
    ),
    "limitations": [
        "Synthetic labels and outcomes.",
        "Probability calibration is not established.",
        "Feature importance and coefficients do not prove causation.",
        "No model is approved for production use.",
        "Reproduction depends on the recorded source files, "
        "software versions, and notebook implementation.",
    ],
}

manifest_path = (
    PROCESSED_DIR / "Day12_Reproducibility_Manifest.json"
)

with manifest_path.open("w", encoding="utf-8") as file:
    json.dump(manifest, file, indent=2)


# ------------------------------------------------------------
# 5. Create Day 12 experiment report
# ------------------------------------------------------------

report_path = (
    REPORTS_DIR / "Day12_Explainable_AI_Governance_Report.md"
)

report_lines = [
    "# Day 12 — Explainable AI and Model Governance",
    "",
    "## Status",
    "",
    "**Technical deliverables completed; business validation pending.**",
    "",
    "## Scope",
    "",
    "The work covers model inventory, coefficient and feature-importance "
    "explanations, exploratory probability calibration, model cards, "
    "and a reproducibility log.",
    "",
    "## Explainability",
    "",
    f"- Explanation records generated: {len(explainability_df)}",
    f"- Models represented in the explanation file: "
    f"{explainability_df['Model_ID'].nunique()}",
    "- Logistic regression coefficients are associations in the fitted model.",
    "- Random Forest impurity importance measures relative model reliance, "
    "not causal effects.",
    "",
    "## Probability calibration",
    "",
]

for _, row in overall_calibration.iterrows():
    report_lines.append(
        f"- **{row['Model_ID']}**: "
        f"{int(row['Test_Records'])} test records; "
        f"observed positive rate {row['Observed_Positive_Rate']:.4f}; "
        f"mean predicted probability "
        f"{row['Mean_Predicted_Probability']:.4f}; "
        f"Brier score {row['Brier_Score']:.4f}; "
        f"log loss {row['Log_Loss']:.4f}."
    )

report_lines.extend([
    "",
    "The calibration results are exploratory. Close agreement between "
    "the overall observed rate and mean predicted probability does not "
    "prove calibration across individual probability ranges.",
    "",
    "## Experiment tracking",
    "",
    f"- Experiment records: {len(experiment_df)}",
    f"- Experiment log: `{experiment_path}`",
    f"- Reproducibility manifest: `{manifest_path}`",
    "",
    "## Data and governance limitations",
    "",
    "- All evaluated datasets are synthetic.",
    "- Real historical business outcomes are needed for real-world validation.",
    "- No model is approved for production.",
    "- Business owners must validate targets, thresholds, and intended use.",
    "- Calibration, fairness, data privacy, and operational monitoring "
    "require additional review before any production deployment.",
    "",
    "## Deliverables",
    "",
    "- `Day12_Model_Inventory.csv`",
    "- `Day12_Model_Explainability.csv`",
    "- `Day12_Probability_Calibration_Checks.csv`",
    "- `Day12_Model_Cards.csv`",
    "- `Day12_Model_Cards_Report.md`",
    "- `Day12_Experiment_Log.csv`",
    "- `Day12_Reproducibility_Manifest.json`",
    "- `Day12_Explainable_AI_Governance_Report.md`",
    "",
])

report_path.write_text(
    "\n".join(report_lines),
    encoding="utf-8",
)


# ------------------------------------------------------------
# 6. Validation
# ------------------------------------------------------------

assert len(experiment_df) == 5
assert experiment_df["Experiment_ID"].is_unique
assert experiment_path.exists()
assert manifest_path.exists()
assert report_path.exists()
assert len(overall_calibration) == 3

print("=" * 75)
print("DAY 12 EXPERIMENT LOG AND GOVERNANCE REPORT")
print("=" * 75)

print("\nExperiment records:", len(experiment_df))
print("Experiment log:", experiment_path)
print("Manifest:", manifest_path)
print("Governance report:", report_path)

print("\nSource dataset checksum status:")
for label, metadata in source_metadata.items():
    status = "FOUND" if metadata["exists"] else "MISSING"
    print(f"{label}: {status}")

print("\nExperiment summary:")
print(
    experiment_df[
        ["Experiment_ID", "Experiment", "Status"]
    ].to_string(index=False)
)

print("\nPASS: Experiment log, manifest, and report created.")
print("Reminder: results use synthetic demonstration data.")

DAY 12 EXPERIMENT LOG AND GOVERNANCE REPORT

Experiment records: 5
Experiment log: D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\processed\Day12_Experiment_Log.csv
Manifest: D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\processed\Day12_Reproducibility_Manifest.json
Governance report: D:\Nexzibyte Solution\nexzibyte-predictive-bi\reports\Day12_Explainable_AI_Governance_Report.md

Source dataset checksum status:
Lead model: FOUND
Quotation model: FOUND
Project risk models: FOUND
Revenue forecast: FOUND

Experiment summary:
Experiment_ID                              Experiment                                    Status
DAY12-EXP-001      Quotation win model explainability                                 Completed
DAY12-EXP-002      Project delay model explainability Completed; calibration concern documented
DAY12-EXP-003   Project effort-overrun explainability                                 Completed
DAY12-EXP-004      Probability calibration assessment          Exploratory asse

In [9]:
from pathlib import Path
import pandas as pd

# Project directories
PROJECT_ROOT = Path(
    r"D:\Nexzibyte Solution\nexzibyte-predictive-bi"
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SYNTHETIC_DIR = PROJECT_ROOT / "data" / "synthetic"
RAW_DIR = PROJECT_ROOT / "data" / "raw"

# Candidate datasets for the fairness assessment
candidate_files = [
    PROCESSED_DIR / "Day5_Feature_Dataset.csv",
    SYNTHETIC_DIR / "Day7_Synthetic_Quotation_Dataset.csv",
    SYNTHETIC_DIR / "Day9_Synthetic_Project_Risk_Dataset.csv",
]

print("=" * 75)
print("DAY 12 — FAIRNESS DATA INVENTORY")
print("=" * 75)

for file_path in candidate_files:
    print(f"\nDataset: {file_path.name}")

    if not file_path.exists():
        print("STATUS: File not found")
        continue

    df = pd.read_csv(file_path)

    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Column names:")

    for column in df.columns:
        print(f"  - {column}")

    print("\nMissing values by column:")

    missing_values = df.isna().sum()
    missing_values = missing_values[missing_values > 0]

    if missing_values.empty:
        print("  None")
    else:
        print(missing_values.to_string())

    print("\nPASS: Dataset inspected.")

DAY 12 — FAIRNESS DATA INVENTORY

Dataset: Day5_Feature_Dataset.csv
Rows: 180
Columns: 15
Column names:
  - Company Name
  - Industry / Sub-industry
  - Founded Year (Standardized)
  - Target Customers / B2B or B2C
  - Employee Count (Numeric)
  - Company Size Score
  - B2B Fit Score
  - Technology Signal Score
  - Business Need Score
  - Recent Business Signal Score
  - Decision-Maker Score
  - Data Quality Status
  - Employee Validation
  - Company Age (Years)
  - Company Age Validation

Missing values by column:
  None

PASS: Dataset inspected.

Dataset: Day7_Synthetic_Quotation_Dataset.csv
Rows: 300
Columns: 12
Column names:
  - Quote_ID
  - Company_ID
  - Quote_Date
  - Quote_Value
  - Discount_Pct
  - Revision_Count
  - Engagement_Count
  - Quote_Status
  - Accepted_Date
  - Loss_Reason
  - Synthetic_Data_Flag
  - Data_Use

Missing values by column:
Accepted_Date    182
Loss_Reason      163

PASS: Dataset inspected.

Dataset: Day9_Synthetic_Project_Risk_Dataset.csv
Rows: 300
Colu

In [10]:
from pathlib import Path
import pandas as pd

# ============================================================
# DAY 12 - FAIRNESS ASSESSMENT PLAN
# ============================================================

PROJECT_ROOT = Path(
    r"D:\Nexzibyte Solution\nexzibyte-predictive-bi"
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

assessment_rows = [
    {
        "Model_ID": "DAY6_LEAD_MODEL",
        "Comparison_Group": "Industry / Sub-industry",
        "Assessment_Type": "Predictive performance by group",
        "Required_Metrics": (
            "Group size, prediction rate, precision, recall, "
            "false-positive rate, false-negative rate"
        ),
        "Data_Availability": "Candidate field available in Day5 feature dataset",
        "Current_Status": "Pending evaluation",
        "Limitation": (
            "Synthetic target; confirm group sizes and connect "
            "the correct test-set predictions to their records."
        ),
    },
    {
        "Model_ID": "DAY6_LEAD_MODEL",
        "Comparison_Group": "B2B versus B2C",
        "Assessment_Type": "Predictive performance by group",
        "Required_Metrics": (
            "Group size, prediction rate, precision, recall, "
            "false-positive rate, false-negative rate"
        ),
        "Data_Availability": "Candidate field available in Day5 feature dataset",
        "Current_Status": "Pending evaluation",
        "Limitation": (
            "Synthetic target; check group sizes and verify that "
            "the model test records retain their group labels."
        ),
    },
    {
        "Model_ID": "DAY7_QUOTE_LOGISTIC",
        "Comparison_Group": "Industry / customer type",
        "Assessment_Type": "Predictive performance by group",
        "Required_Metrics": (
            "Group size, prediction rate, precision, recall, "
            "false-positive rate, false-negative rate"
        ),
        "Data_Availability": "Not directly available in quotation dataset",
        "Current_Status": "Blocked pending data mapping",
        "Limitation": (
            "Verify a reliable company identifier and valid mapping "
            "before joining company attributes to quotations."
        ),
    },
    {
        "Model_ID": "DAY9_PROJECT_DELAY_RF",
        "Comparison_Group": "Relevant operational project groups",
        "Assessment_Type": "Predictive performance by group",
        "Required_Metrics": (
            "Group size, prediction rate, precision, recall, "
            "false-positive rate, false-negative rate"
        ),
        "Data_Availability": "Suitable grouping variable not yet approved",
        "Current_Status": "Pending group definition",
        "Limitation": (
            "Project characteristics are not automatically protected "
            "groups and do not establish protected-group fairness."
        ),
    },
    {
        "Model_ID": "DAY9_EFFORT_OVERRUN_LOGISTIC",
        "Comparison_Group": "Relevant operational project groups",
        "Assessment_Type": "Predictive performance by group",
        "Required_Metrics": (
            "Group size, prediction rate, precision, recall, "
            "false-positive rate, false-negative rate"
        ),
        "Data_Availability": "Suitable grouping variable not yet approved",
        "Current_Status": "Pending group definition",
        "Limitation": (
            "Project characteristics are not automatically protected "
            "groups and do not establish protected-group fairness."
        ),
    },
]

fairness_plan_df = pd.DataFrame(assessment_rows)

plan_path = (
    PROCESSED_DIR / "Day12_Fairness_Assessment_Plan.csv"
)

fairness_plan_df.to_csv(plan_path, index=False)

report_path = (
    REPORTS_DIR / "Day12_Fairness_Assessment_Report.md"
)

report_lines = [
    "# Day 12 — Preliminary Fairness Assessment",
    "",
    "## Overall status",
    "",
    "**Assessment plan documented; empirical fairness evaluation pending.**",
    "",
    "The current datasets are synthetic. Results from them cannot establish "
    "fairness in real business operations.",
    "",
    "## Assessment approach",
    "",
    "1. Confirm valid group definitions and their business justification.",
    "2. Verify sample sizes and outcome availability for each group.",
    "3. Evaluate prediction rates and relevant error metrics by group.",
    "4. Investigate material differences instead of assuming that a "
    "difference proves unfairness.",
    "5. Document uncertainty, limitations, and the business owner's review.",
    "",
    "## Current limitations",
    "",
    "- The lead dataset has candidate industry and customer-type fields.",
    "- The quotation dataset requires verified mapping before company "
    "attributes can be joined.",
    "- Suitable project comparison groups have not yet been approved.",
    "- No protected-group fairness conclusion has been established.",
    "- No model is approved for production use.",
    "",
    "## Decisions still required",
    "",
    "- Approve appropriate comparison groups and evaluation criteria.",
    "- Confirm sufficient observations per group.",
    "- Link each test prediction to the correct original record.",
    "- Obtain suitable, authorized data for any real-world fairness review.",
    "",
    "## Assessment register",
    "",
    "| Model | Comparison group | Status |",
    "|---|---|---|",
]

for _, row in fairness_plan_df.iterrows():
    report_lines.append(
        f"| {row['Model_ID']} | {row['Comparison_Group']} "
        f"| {row['Current_Status']} |"
    )

report_lines.extend([
    "",
    "## Conclusion",
    "",
    "This deliverable records the assessment plan, not a completed "
    "fairness test. Fairness remains unverified until the required "
    "group-level evaluation is performed and reviewed.",
    "",
])

report_path.write_text(
    "\n".join(report_lines),
    encoding="utf-8",
)

assert len(fairness_plan_df) == 5
assert plan_path.exists()
assert report_path.exists()

print("=" * 75)
print("DAY 12 — FAIRNESS ASSESSMENT PLAN")
print("=" * 75)

print("Assessment entries:", len(fairness_plan_df))
print("Plan CSV:", plan_path)
print("Report:", report_path)

print("\nStatus summary:")
print(
    fairness_plan_df[
        ["Model_ID", "Comparison_Group", "Current_Status"]
    ].to_string(index=False)
)

print("\nPASS: Fairness assessment plan saved.")
print("Note: No empirical fairness conclusion has been made.")

DAY 12 — FAIRNESS ASSESSMENT PLAN
Assessment entries: 5
Plan CSV: D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\processed\Day12_Fairness_Assessment_Plan.csv
Report: D:\Nexzibyte Solution\nexzibyte-predictive-bi\reports\Day12_Fairness_Assessment_Report.md

Status summary:
                    Model_ID                    Comparison_Group               Current_Status
             DAY6_LEAD_MODEL             Industry / Sub-industry           Pending evaluation
             DAY6_LEAD_MODEL                      B2B versus B2C           Pending evaluation
         DAY7_QUOTE_LOGISTIC            Industry / customer type Blocked pending data mapping
       DAY9_PROJECT_DELAY_RF Relevant operational project groups     Pending group definition
DAY9_EFFORT_OVERRUN_LOGISTIC Relevant operational project groups     Pending group definition

PASS: Fairness assessment plan saved.
Note: No empirical fairness conclusion has been made.


DAY 12 — LEAD MODEL GROUP INSPECTION

Group field: Industry / Sub-industry
Number of groups: 19

Group counts:
Industry / Sub-industry
B2B SaaS / Enterprise Software             72
AI / ML / GenAI                            52
FinTech / Payments                         39
Customer Engagement SaaS / MarTech          2
SaaS / Business Software / AI               1
API Development / B2B SaaS                  1
Software Testing / DevOps SaaS              1
FinTech / Digital Payments                  1
FinTech / Brokerage Technology              1
SaaS / Subscription Billing                 1
Enterprise Software / SaaS                  1
HRTech / Enterprise SaaS                    1
B2B SaaS / Digital Adoption                 1
CRM / Sales & Marketing Automation SaaS     1
FinTech / Enterprise SaaS / AI              1
Developer Infrastructure / Data APIs        1
Enterprise AI / B2B SaaS                    1
B2B Commerce / Supply Chain                 1
Cloud Data Protection / Cybersecurity

In [13]:
from pathlib import Path

project_dir = Path(
    r"D:\Nexzibyte Solution\nexzibyte-predictive-bi"
)

processed_dir = project_dir / "data" / "processed"

print("Processed folder exists:", processed_dir.exists())

if processed_dir.exists():
    print("\nRelevant CSV files:")

    for file in sorted(processed_dir.glob("*.csv")):
        if "day6" in file.name.lower() or "lead" in file.name.lower():
            print(file.name)

Processed folder exists: True

Relevant CSV files:


In [14]:
from pathlib import Path

project_dir = Path(
    r"D:\Nexzibyte Solution\nexzibyte-predictive-bi"
)

print("Searching for CSV files containing 'scored'...")

matches = []

for file in project_dir.rglob("*.csv"):
    if "scored" in file.name.lower():
        matches.append(file)

if matches:
    for file in matches:
        print(file.relative_to(project_dir))
else:
    print("No scored CSV files found.")

print("\nSearching for Day 6 model output files...")

for file in project_dir.rglob("*.csv"):
    name = file.name.lower()

    if "day6" in name or "lead_model" in name:
        print(file.relative_to(project_dir))

Searching for CSV files containing 'scored'...
data\raw\NexziByte_Final_Scored.csv
data\synthetic\Day6_Synthetic_Scored_Output.csv

Searching for Day 6 model output files...
data\synthetic\Day6_Synthetic_Baseline_Metrics.csv
data\synthetic\Day6_Synthetic_Priority_Queue.csv
data\synthetic\Day6_Synthetic_Scored_Output.csv


In [15]:
from pathlib import Path
import pandas as pd

project_dir = Path(
    r"D:\Nexzibyte Solution\nexzibyte-predictive-bi"
)

scored_path = (
    project_dir
    / "data"
    / "synthetic"
    / "Day6_Synthetic_Scored_Output.csv"
)

scored = pd.read_csv(scored_path)

print("File loaded successfully.")
print("Rows:", len(scored))
print("\nColumns:")
print(scored.columns.tolist())

print("\nFirst 5 rows:")
display(scored.head())

File loaded successfully.
Rows: 45

Columns:
['Source_Row_Index', 'Industry / Sub-industry', 'Target Customers / B2B or B2C', 'Founded Year (Standardized)', 'Employee Count (Numeric)', 'Company Age (Years)', 'Synthetic_Actual_Label', 'Synthetic_Predicted_Label', 'Synthetic_Predicted_Probability', 'Data Status']

First 5 rows:


,Source_Row_Index,Industry / Sub-industry,Target Customers / B2B or B2C,Founded Year (Standardized),Employee Count (Numeric),Company Age (Years),Synthetic_Actual_Label,Synthetic_Predicted_Label,Synthetic_Predicted_Probability,Data Status
0,119,AI / ML / GenAI,B2B — businesses and enterprise customers,2020,100,6,0,0,0.200986,SYNTHETIC DEMONSTRATION ONLY
1,63,B2B SaaS / Enterprise Software,B2B2C — employers/recruiters plus students and...,2017,500,9,1,0,0.305988,SYNTHETIC DEMONSTRATION ONLY
2,167,FinTech / Payments,B2C + B2B — consumers and businesses seeking s...,2015,500,11,1,0,0.321859,SYNTHETIC DEMONSTRATION ONLY
3,85,B2B SaaS / Enterprise Software,"B2B/B2G — hospitals, diagnostic centers, scree...",2016,150,10,0,0,0.292753,SYNTHETIC DEMONSTRATION ONLY
4,154,FinTech / Payments,B2C — consumers seeking digital banking and mo...,2019,500,7,0,0,0.430414,SYNTHETIC DEMONSTRATION ONLY


In [16]:
# ============================================================
# DAY 12 — PRELIMINARY LEAD MODEL FAIRNESS ASSESSMENT
# Synthetic demonstration data only
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.metrics import (
    precision_score,
    recall_score,
    confusion_matrix
)

# 1. Define project paths
project_dir = Path(
    r"D:\Nexzibyte Solution\nexzibyte-predictive-bi"
)

scored_path = (
    project_dir
    / "data"
    / "synthetic"
    / "Day6_Synthetic_Scored_Output.csv"
)

processed_dir = project_dir / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

# 2. Load the existing test predictions
if not scored_path.exists():
    raise FileNotFoundError(
        f"Prediction file not found: {scored_path}"
    )

scored = pd.read_csv(scored_path)

required_columns = [
    "Source_Row_Index",
    "Industry / Sub-industry",
    "Target Customers / B2B or B2C",
    "Synthetic_Actual_Label",
    "Synthetic_Predicted_Label",
    "Synthetic_Predicted_Probability"
]

missing_columns = [
    col for col in required_columns
    if col not in scored.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

if len(scored) != 45:
    raise ValueError(
        f"Expected 45 test records, found {len(scored)}. "
        "Stop and investigate."
    )

print("Loaded original test predictions:", len(scored))

# 3. Define metrics for each group
def calculate_group_metrics(group, group_field):

    actual = group["Synthetic_Actual_Label"].astype(int)
    predicted = group["Synthetic_Predicted_Label"].astype(int)

    tn, fp, fn, tp = confusion_matrix(
        actual,
        predicted,
        labels=[0, 1]
    ).ravel()

    negative_count = tn + fp
    positive_count = tp + fn

    return {
        "Group Field": group_field,
        "Group": str(group.name),
        "Records": len(group),

        "Actual Positive Rate": actual.mean(),

        "Predicted Positive Rate": predicted.mean(),

        "Mean Predicted Probability": (
            group["Synthetic_Predicted_Probability"].mean()
        ),

        "Precision": precision_score(
            actual,
            predicted,
            zero_division=0
        ),

        "Recall": recall_score(
            actual,
            predicted,
            zero_division=0
        ),

        "False Positive Rate": (
            fp / negative_count
            if negative_count > 0
            else np.nan
        ),

        "False Negative Rate": (
            fn / positive_count
            if positive_count > 0
            else np.nan
        ),

        "True Positives": tp,
        "False Positives": fp,
        "True Negatives": tn,
        "False Negatives": fn,

        "Data Status": "SYNTHETIC — EXPLORATORY ONLY",

        "Interpretation Warning": (
            "Very small sample; metrics are unstable"
            if len(group) < 10
            else "Synthetic labels; interpret cautiously"
        )
    }


# 4. Assess industry groups
results = []

industry_field = "Industry / Sub-industry"

for group_name, group in scored.groupby(
    industry_field,
    dropna=False
):

    group = group.copy()
    group.name = group_name

    results.append(
        calculate_group_metrics(
            group,
            industry_field
        )
    )

industry_fairness = pd.DataFrame(results)

# 5. Save the industry assessment
industry_output = (
    processed_dir / "Day12_Lead_Model_Group_Fairness.csv"
)

industry_fairness.to_csv(
    industry_output,
    index=False
)

print("\nINDUSTRY GROUP ASSESSMENT")
print("Groups assessed:", len(industry_fairness))
print("Saved to:", industry_output)

display(
    industry_fairness.sort_values(
        ["Records", "Group"],
        ascending=[False, True]
    ).reset_index(drop=True)
)

# 6. Assess exact customer-type groups separately
# Most groups may contain only one record.
customer_field = "Target Customers / B2B or B2C"

customer_results = []

for group_name, group in scored.groupby(
    customer_field,
    dropna=False
):

    group = group.copy()
    group.name = group_name

    customer_results.append(
        calculate_group_metrics(
            group,
            customer_field
        )
    )

customer_fairness = pd.DataFrame(customer_results)

customer_output = (
    processed_dir
    / "Day12_Lead_Model_Customer_Type_Fairness.csv"
)

customer_fairness.to_csv(
    customer_output,
    index=False
)

print("\nCUSTOMER-TYPE GROUP ASSESSMENT")
print("Groups assessed:", len(customer_fairness))
print("Saved to:", customer_output)

display(
    customer_fairness.sort_values(
        ["Records", "Group"],
        ascending=[False, True]
    ).reset_index(drop=True)
)

print("\nASSESSMENT COMPLETE")
print("All results use synthetic demonstration labels.")
print("These results do not establish real-world model fairness.")

Loaded original test predictions: 45

INDUSTRY GROUP ASSESSMENT
Groups assessed: 7
Saved to: D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\processed\Day12_Lead_Model_Group_Fairness.csv


,Group Field,Group,Records,Actual Positive Rate,Predicted Positive Rate,Mean Predicted Probability,Precision,Recall,False Positive Rate,False Negative Rate,True Positives,False Positives,True Negatives,False Negatives,Data Status,Interpretation Warning
0,Industry / Sub-industry,AI / ML / GenAI,14,0.285714,0.071429,0.253619,0.0,0.00,0.100000,1.00,0,1,9,4,SYNTHETIC — EXPLORATORY ONLY,Synthetic labels; interpret cautiously
1,Industry / Sub-industry,B2B SaaS / Enterprise Software,14,0.357143,0.000000,0.305376,0.0,0.00,0.000000,1.00,0,0,9,5,SYNTHETIC — EXPLORATORY ONLY,Synthetic labels; interpret cautiously
2,Industry / Sub-industry,FinTech / Payments,13,0.307692,0.153846,0.397198,0.5,0.25,0.111111,0.75,1,1,8,3,SYNTHETIC — EXPLORATORY ONLY,Synthetic labels; interpret cautiously
3,Industry / Sub-industry,B2B SaaS / Digital Adoption,1,1.000000,0.000000,0.262563,0.0,0.00,NaN,1.00,0,0,0,1,SYNTHETIC — EXPLORATORY ONLY,Very small sample; metrics are unstable
4,Industry / Sub-industry,CRM / Sales & Marketing Automation SaaS,1,1.000000,0.000000,0.322885,0.0,0.00,NaN,1.00,0,0,0,1,SYNTHETIC — EXPLORATORY ONLY,Very small sample; metrics are unstable
5,Industry / Sub-industry,Cloud Data Protection / Cybersecurity,1,1.000000,0.000000,0.240069,0.0,0.00,NaN,1.00,0,0,0,1,SYNTHETIC — EXPLORATORY ONLY,Very small sample; metrics are unstable
6,Industry / Sub-industry,Customer Engagement SaaS / MarTech,1,0.000000,0.000000,0.224645,0.0,0.00,0.000000,NaN,0,0,1,0,SYNTHETIC — EXPLORATORY ONLY,Very small sample; metrics are unstable



CUSTOMER-TYPE GROUP ASSESSMENT
Groups assessed: 44
Saved to: D:\Nexzibyte Solution\nexzibyte-predictive-bi\data\processed\Day12_Lead_Model_Customer_Type_Fairness.csv


,Group Field,Group,Records,Actual Positive Rate,Predicted Positive Rate,Mean Predicted Probability,Precision,Recall,False Positive Rate,False Negative Rate,True Positives,False Positives,True Negatives,False Negatives,Data Status,Interpretation Warning
0,Target Customers / B2B or B2C,B2B / enterprises,2,1.0,0.0,0.281477,0.0,0.0,NaN,1.0,0,0,0,2,SYNTHETIC — EXPLORATORY ONLY,Very small sample; metrics are unstable
1,Target Customers / B2B or B2C,B2B & B2C / consumers and merchants,1,0.0,1.0,0.579250,0.0,0.0,1.0,NaN,0,1,0,0,SYNTHETIC — EXPLORATORY ONLY,Very small sample; metrics are unstable
2,Target Customers / B2B or B2C,B2B / businesses and enterprises,1,1.0,1.0,0.605860,1.0,1.0,NaN,0.0,1,0,0,0,SYNTHETIC — EXPLORATORY ONLY,Very small sample; metrics are unstable
3,Target Customers / B2B or B2C,B2B / consumer brands and enterprises,1,0.0,0.0,0.224645,0.0,0.0,0.0,NaN,0,0,1,0,SYNTHETIC — EXPLORATORY ONLY,Very small sample; metrics are unstable
4,Target Customers / B2B or B2C,B2B / global enterprises,1,1.0,0.0,0.262563,0.0,0.0,NaN,1.0,0,0,0,1,SYNTHETIC — EXPLORATORY ONLY,Very small sample; metrics are unstable
5,Target Customers / B2B or B2C,"B2B — AI companies, enterprises, developers an...",1,0.0,0.0,0.207015,0.0,0.0,0.0,NaN,0,0,1,0,SYNTHETIC — EXPLORATORY ONLY,Very small sample; metrics are unstable
6,Target Customers / B2B or B2C,B2B — SMEs and businesses purchasing raw mater...,1,1.0,0.0,0.455435,0.0,0.0,NaN,1.0,0,0,0,1,SYNTHETIC — EXPLORATORY ONLY,Very small sample; metrics are unstable
7,Target Customers / B2B or B2C,B2B — SMEs and small-business owners,1,0.0,0.0,0.352212,0.0,0.0,0.0,NaN,0,0,1,0,SYNTHETIC — EXPLORATORY ONLY,Very small sample; metrics are unstable
8,Target Customers / B2B or B2C,"B2B — SMEs, businesses, banks and fintech ecos...",1,0.0,0.0,0.313690,0.0,0.0,0.0,NaN,0,0,1,0,SYNTHETIC — EXPLORATORY ONLY,Very small sample; metrics are unstable
9,Target Customers / B2B or B2C,"B2B — advertisers, brands, app developers and ...",1,1.0,0.0,0.369214,0.0,0.0,NaN,1.0,0,0,0,1,SYNTHETIC — EXPLORATORY ONLY,Very small sample; metrics are unstable



ASSESSMENT COMPLETE
All results use synthetic demonstration labels.
These results do not establish real-world model fairness.
